In [ ]:
vocab = "$abcdefghijklmnopqrstuvwxyz"
vocab_size = len(vocab)

ch_to_i = {char: i for i, char in enumerate(vocab)}
i_to_ch = {i: char for i, char in enumerate(vocab)}

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

equal_probs = F.softmax(torch.ones(vocab_size), dim=0)
for i in range(5):
    generated = ""
    while True:
        rand_int = torch.multinomial(equal_probs, 1).item()
        rand_char = i_to_ch[rand_int]
        if rand_char == "$":
            break

        generated += rand_char

    print(f"name {i}: {generated}")

In [ ]:
names = []
# edits each name into $<name>$  
with open('data/names_2022.txt', 'r') as file:
    for line in file:
        name, _, _= line.lower().strip().split(',')
        names.append("$" + name + "$")
len(names)


In [ ]:
bigram = torch.zeros((vocab_size, vocab_size))
total = 0
for name in names:
    for ch1, ch2 in zip(name, name[1:]):
        ch1_int = ch_to_i[ch1]
        ch2_int = ch_to_i[ch2]
        bigram[ch1_int][ch2_int] += 1
        total += 1
bigram /= total
    

In [ ]:
for i in range(5):
    generated = "$"
    while True:
        bigram_probs = bigram[ch_to_i[generated[-1]]]
        sampled_char = i_to_ch[
            torch.multinomial(bigram_probs, 1).item()
        ]
        if sampled_char == "$":
            break
        generated += sampled_char
    print(f"name {i}: {generated[1:]}")


In [ ]:
example_name = "$ada$"
encode = lambda word: torch.tensor([ch_to_i[c] for c in word])
decode = lambda tensor_i: ''.join(i_to_ch[i.item()] for i in tensor_i)
print(encode(example_name))
print(decode(encode(example_name)))

name_indices = [encode(name) for name in names]
target_indices = [name_index[1:] for name_index in name_indices]

In [ ]:
from torch.nn.utils.rnn import pad_sequence
X = pad_sequence(name_indices, batch_first=True, padding_value=0)
max_name_length = max(len(name) for name in names)
target_indices.append(torch.empty((max_name_length), dtype=torch.long)) # adds a new dummy tensor into target_indices so that max lenght of target will be 11
Y = pad_sequence(target_indices, batch_first=True, padding_value=-1)[:-1]
print(X[0])
print(Y[0])

In [ ]:
def get_batch(batch_size=64):
    """creates a random batch of input and labels"""
    random_idx = torch.randint(0, X.size(0), (batch_size,)) # creating a random index including dimensions of the padded sequence and the total amount of names. eg; 31915
    print(X.shape) 
    print(X.size(0)) 
    print(random_idx)
    inputs = X[random_idx]
    labels = Y[random_idx]
    return inputs, labels
inputs, labels = get_batch(3)
print(inputs)
print(inputs.shape)
print(labels)


In [ ]:
embedding_dim = 3 # arbitrary
embedding = nn.Embedding(vocab_size, embedding_dim)
example_input = torch.tensor([1,1,0,2])
input_emb = embedding(example_input)
print(input_emb.shape) # 4 rows for 4 embeddings with 3 dimensions (defined)
input_emb

In [ ]:
class SequenceMLP(nn.Module):
    def __init__(self, vocab_size, max_sequence_length, embedding_dim, hidden_dim=32):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_sequence_length = max_sequence_length
        self.embedding_dim = embedding_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim*max_sequence_length, hidden_dim) # flattening the padded matrix into a vector
        self.relu = nn.ReLU()
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.shape # x = inputs; here: batch_size=64; seq_len is 17 
        sequence_embeddings = torch.zeros(
            batch_size, seq_len, 
            self.max_sequence_length*self.embedding_dim
        ) # creating flattened matrix

        for i in range(seq_len): # seq_len is 17 here
            subsequence = torch.zeros(
                batch_size, # 64
                self.max_sequence_length, # 17
                dtype=torch.int
            )
            prefix = x[:, :i+1] # slices and selects up to the i+1h index
            subsequence[:, :i+1] = prefix # stores the same thing as prefix with the rest of the values being zero
            emb = self.embedding(subsequence)  # lookd up the embedding value of the subsequence and stores it
            sequence_embeddings[:, i, :] = emb.view(batch_size, -1)
        x = self.linear(sequence_embeddings)
        x = self.relu(x)
        x = self.out(x)
        return x

embedding_dim = 3
max_sequence_length = X.shape[1] # 17
model = SequenceMLP(vocab_size, max_sequence_length, embedding_dim)



In [ ]:
import torch.optim as optim

def train(model, optimizer, num_steps=10_001, loss_report_interval=1_000):
    losses = []
    for i in range(1, num_steps):
        inputs, labels = get_batch() # gets the inputs and labels randomly created with get_batch()
        optimizer.zero_grad()
        logits = model(inputs)
        loss = F.cross_entropy(
            input=logits.view(-1, logits.shape[-1]),
            target=labels.view(-1), 
            ignore_index=-1
        )    
        losses.append(loss.item())
        if i % loss_report_interval == 0:
            print(f'Average loss at step {i}: {sum(losses[
                -loss_report_interval:]) / loss_report_interval:.4f}')
        loss.backward()
        optimizer.step()

optimizer = optim.SGD(model.parameters(), lr=0.1)

In [ ]:
train(model, optimizer)

In [ ]:
x = torch.tensor((1,2,3,4,5))
for i in range(4):
    print(x[:i+1])
x[:]